# Questão 2
A estratégia adotada no arquivo foi dividir o trabalho em etapas:

1. conexão com o ambiente Snowflake;
2. leitura da tabela bruta;
3. conversão dos dados para pandas;
4. inspeção dos valores originais da categoria;
5. normalização textual;
6. aplicação de um dicionário de equivalência;
7. validação do resultado final.

Ao final, todas as variações são consolidadas em categorias padronizadas, facilitando análises posteriores e melhorando a qualidade do dado.





# Tarefa 1 - Padronização da Coluna `ACTUAL_CATEGORY` a partir de Dados no Snowflake

Foi implementado um processo de extração, tratamento e padronização da coluna ACTUAL_CATEGORY no Snowflake, visando corrigir inconsistências de escrita e formatação.

Os dados foram acessados via Snowpark, convertidos para pandas e analisados para identificar variações. Em seguida, aplicou-se normalização textual e um dicionário de mapeamento, consolidando as categorias em eletrônicos, propulsão e ancoragem.

Por fim, foi realizada a validação dos dados, garantindo consistência e qualidade para análises confiáveis.

In [1]:
import pandas as pd
import os 

from dotenv import load_dotenv
load_dotenv()

from snowflake.snowpark import Session

conn = {
    "account": os.getenv("account"),
    "user": os.getenv("user"),
    "password": os.getenv("password"),
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "LH_NAUTICALS"
}

session = Session.builder.configs(conn).create()


df_snowpark = session.table("LH_NAUTICALS.QUESTAO2.PRODUTOS_RAW")
df_pandas = df_snowpark.to_pandas()

In [2]:
df_pandas["ACTUAL_CATEGORY"].unique()

array(['ELETRONICOS', 'E L E T R Ô N I C O S', 'Eletrunicos',
       'Eletronicoz', 'eLeTrÔnIcOs', 'eletrônicos', 'Eletrônicos',
       'Eletroniscos', 'Eletronicos', 'eletronicos', 'EletrônicoS',
       'ELEtRÔNICOS', 'PROPULSAO', 'Propulção', 'Prop', 'Propulssão',
       'propulsao', 'P R O P U L S Ã O', 'Propução', 'propulsão',
       'pRoPuLsÃo', 'Propulçao', 'Propulsam', 'PrOpUlSãO', 'Ancoragem',
       'AnCoRaGeM', 'Encoragem', 'Ancoraguem', 'Ancorajm', 'AncorageM',
       'A N C O R A G E M', 'ANCORAGEM', 'aNcOrAgEm', 'Ancorajem',
       'Encoragi', 'ancoragem', 'Ancorajen', 'AncorajeM', 'Ancoragen'],
      dtype=object)

In [3]:
df_pandas["ACTUAL_CATEGORY"] = (
    df_pandas["ACTUAL_CATEGORY"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(" ", "", regex=False)  # remove espaços tipo "E L E T..."
)


mapeamento_categorias = {
    # ELETRÔNICOS
    'eletronicos': 'eletrônicos',
    'eletrônicoS'.lower(): 'eletrônicos',
    'eletrônicos': 'eletrônicos',
    'e l e t r ô n i c o s': 'eletrônicos',
    'eletrunicos': 'eletrônicos',
    'eletronicoz': 'eletrônicos',
    'eletroniscos': 'eletrônicos',
    'eletronicos': 'eletrônicos',
    'eletrônicos': 'eletrônicos',
    
    # PROPULSÃO
    'propulsao': 'propulsão',
    'propulção': 'propulsão',
    'prop': 'propulsão',
    'propulssão': 'propulsão',
    'p r o p u l s ã o': 'propulsão',
    'propução': 'propulsão',
    'propulsão': 'propulsão',
    'propulçao': 'propulsão',
    'propulsam': 'propulsão',
    
    # ANCORAGEM
    'ancoragem': 'ancoragem',
    'ancoraguem': 'ancoragem',
    'ancorajm': 'ancoragem',
    'ancorajem': 'ancoragem',
    'ancorajen': 'ancoragem',
    'ancorajem': 'ancoragem',
    'ancoragen': 'ancoragem',
    'encoragem': 'ancoragem',
    'encoragi': 'ancoragem',
    'a n c o r a g e m': 'ancoragem'
}

df_pandas["ACTUAL_CATEGORY"] = df_pandas["ACTUAL_CATEGORY"].replace(mapeamento_categorias)
df_pandas["ACTUAL_CATEGORY"].unique()




array(['eletrônicos', 'propulsão', 'ancoragem'], dtype=object)

In [4]:
df_pandas.head(70)

,NAME,PRICE,CODE,ACTUAL_CATEGORY
0,Transponder AIS Maré Magnum,R$ 33122.52,1,eletrônicos
1,Transponder Furuno Marlin,R$ 13998.15,2,eletrônicos
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,eletrônicos
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,eletrônicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,eletrônicos
...,...,...,...,...
65,Motor Diesel Yanmar Velocity 37HP,R$ 102221.97,62,propulsão
66,Motor Elétrico Torqeedo Ion Orca Vox 186HP,R$ 139957.67,63,propulsão
67,Motor Elétrico Yamaha Swift Drift Current 265HP,R$ 111505.55,64,propulsão
68,Motor de Popa Torqeedo Nautic Swift 223HP,R$ 25557.7,65,propulsão


# Tarefa 2 - tratamento de tipo para numeral.

Foi realizado o tratamento e padronização da coluna PRICE, visando garantir sua correta utilização em análises quantitativas. Inicialmente, identificou-se que os dados estavam em formato textual devido à presença de caracteres não numéricos, como o símbolo de moeda.

Em seguida, aplicou-se um processo de limpeza e transformação dos dados, incluindo remoção de caracteres indesejados, padronização das strings e conversão para o tipo numérico (float).

Por fim, foram realizadas etapas de validação estrutural e visual, assegurando que a coluna passou a apresentar o formato adequado e consistente para operações matemáticas e análises.



In [5]:
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   NAME             157 non-null    object
 1   PRICE            157 non-null    object
 2   CODE             157 non-null    int16 
 3   ACTUAL_CATEGORY  157 non-null    object
dtypes: int16(1), object(3)
memory usage: 4.1+ KB


In [6]:
df_pandas["PRICE"] = (
    df_pandas["PRICE"]
    .astype(str)
    .str.replace("R$ ", "", regex=False)  
    .str.strip()
)


df_pandas["PRICE"] = df_pandas["PRICE"].astype(float)

df_pandas.info()
df_pandas.head(5)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   NAME             157 non-null    object 
 1   PRICE            157 non-null    float64
 2   CODE             157 non-null    int16  
 3   ACTUAL_CATEGORY  157 non-null    object 
dtypes: float64(1), int16(1), object(2)
memory usage: 4.1+ KB


,NAME,PRICE,CODE,ACTUAL_CATEGORY
0,Transponder AIS Maré Magnum,33122.52,1,eletrônicos
1,Transponder Furuno Marlin,13998.15,2,eletrônicos
2,Radar Furuno Pulse Leviathan,9024.19,3,eletrônicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,eletrônicos
4,Piloto Automático Furuno Storm,23669.01,5,eletrônicos


# Tarefa 3 - Remoção de dados duplicados

Foi realizada a identificação e remoção de registros duplicados no DataFrame, com o objetivo de garantir a qualidade e a consistência dos dados.

Inicialmente, os duplicados foram identificados e quantificados utilizando funções específicas do pandas. Em seguida, aplicou-se o método drop_duplicates(), removendo as ocorrências repetidas e gerando um novo DataFrame tratado.

Por fim, foi realizada uma validação pós-processamento, confirmando a eliminação completa dos duplicados e assegurando a integridade da base de dados.



In [7]:
df_pandas[df_pandas.duplicated()]

,NAME,PRICE,CODE,ACTUAL_CATEGORY
37,GPS Lowrance Evo Storm Drift,6067.71,37,eletrônicos
63,Motor Diesel Yanmar Velocity 37HP,102221.97,62,propulsão
64,Motor Diesel Yanmar Velocity 37HP,102221.97,62,propulsão
65,Motor Diesel Yanmar Velocity 37HP,102221.97,62,propulsão
132,Cabo de Nylon Delta Velocity Core Mako,1549.35,127,ancoragem
150,Boia de Arqueamento Delta Nexus,4349.86,145,ancoragem
151,Boia de Arqueamento Delta Nexus,4349.86,145,ancoragem


In [8]:
valores_duplicados_antes = df_pandas.duplicated().sum()


df_pandas_tratados = df_pandas.drop_duplicates()
print("Duplicados removidos:", valores_duplicados_antes)


valores_duplicados_depois = df_pandas_tratados.duplicated().sum()
print("Numero de duplicados após o tratamento:", valores_duplicados_depois)

Duplicados removidos: 7
Numero de duplicados após o tratamento: 0


## Conclusão

Esse processo é fundamental em pipelines de dados, pois a presença de duplicatas pode distorcer análises, gerar contagens incorretas e comprometer a confiabilidade dos resultados. Ao remover os registros duplicados, garantimos uma base mais consistente e adequada para análises futuras.